In [ ]:
!pip install -qU langchain langchain-community langchain-huggingface
!pip install -qU langchain-chroma langchain-text-splitters
!pip install -qU sentence-transformers chromadb pypdf
!pip install -qU langchain-groq gradio

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

import gradio as gr
import json

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

In [ ]:
vector_store = Chroma(
    collection_name="ExamAssistant_Collection",
    embedding_function=embedding_model,
    persist_directory="./chroma_langchain_db"
)

In [ ]:
documents_uploaded = False

In [ ]:
def process_pdfs(files):

    global documents_uploaded

    if not files:
        return "Please upload at least one PDF."

    all_documents = []

    # Load PDFs
    for file in files:

        loader = PyPDFLoader(file.name)

        docs = loader.load()

        all_documents.extend(docs)

    # Split documents using RecursiveCharacterTextSplitter
    all_splits = text_splitter.split_documents(
        all_documents
    )

    # Store chunks in ChromaDB
    vector_store.add_documents(
        documents=all_splits
    )

    documents_uploaded = True

    return (
        f"Successfully processed {len(files)} PDF(s).\n"
        f"Created {len(all_splits)} chunks."
    )

In [ ]:
def retrieve_context(user_query, k=4):

    source_docs = vector_store.similarity_search(
        user_query,
        k=k
    )

    context = "\n\n".join(
        doc.page_content
        for doc in source_docs
    )

    return context, source_docs

In [ ]:
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")

model = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=api_key,
    temperature=0
)

In [ ]:
def docu_chat(user_query):

    global documents_uploaded

    # Do not answer without uploaded documents
    if not documents_uploaded:

        return {
            "answer": "Please upload and process PDF documents first.",
            "source_documents": [],
            "context_used": ""
        }

    # Retrieve relevant chunks
    context, source_docs = retrieve_context(
        user_query,
        k=4
    )

    system_message = f"""
You are an AI Exam Question Bank Assistant.

You MUST answer using ONLY the information
from the retrieved context.

If the answer cannot be found in the context,
say:

"I could not find the answer in the uploaded documents."

Do not use outside knowledge.
Do not make up information.

IMPORTANT RESPONSE RULES:

If the user asks for important questions:
Return ONLY the questions.
Do not provide answers or explanations.

If the user asks for repeated questions:
Return ONLY the repeated questions.

If the user asks for topic-wise questions:
Return ONLY the relevant questions.

If the user asks for an explanation or answer:
Provide the answer using only the retrieved context.

RETRIEVED CONTEXT:
{context}
"""

    response = model.invoke([
        ("system", system_message),
        ("human", user_query)
    ])

    return {
        "answer": response.content,
        "source_documents": source_docs,
        "context_used": context
    }

In [ ]:
def ask_question(query):

    if not documents_uploaded:
        return "Please upload and process your PDFs first."

    if not query.strip():
        return "Please enter a question."

    result = docu_chat(query)

    return result["answer"]

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("# 📚 AI Exam Question Bank Assistant")

    pdfs = gr.File(
        label="Upload PDFs",
        file_types=[".pdf"],
        file_count="multiple"
    )

    process_btn = gr.Button("Process PDFs")

    status = gr.Textbox(label="Status")

    process_btn.click(
        process_pdfs,
        inputs=pdfs,
        outputs=status
    )

    query = gr.Textbox(
        label="Ask your question",
        placeholder="Give me the important questions related to overfitting."
    )

    ask_btn = gr.Button("Ask")

    answer = gr.Textbox(
        label="Answer",
        lines=10
    )

    ask_btn.click(
        ask_question,
        inputs=query,
        outputs=answer
    )


demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a6424eedffa4c93f80.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def llm_judge(question, context, answer):

    judge_prompt = f"""
You are a strict evaluator for a RAG-based
AI Exam Question Bank Assistant.

QUESTION:
{question}

RETRIEVED CONTEXT:
{context}

GENERATED ANSWER:
{answer}

Evaluate the response from 1 to 5.

1. Faithfulness:
Is the answer supported by the retrieved context?

2. Answer Relevance:
Does the answer directly answer the user's request?

3. Context Relevance:
Is the retrieved context relevant to the question?

Return ONLY valid JSON.

Format:

{{
    "faithfulness": 1,
    "answer_relevance": 1,
    "context_relevance": 1
}}
"""

    response = model.invoke([
        ("system", "You are a strict RAG evaluator. Return only JSON."),
        ("human", judge_prompt)
    ])

    return json.loads(response.content)

In [ ]:
evaluation_tasks = [

    "Give the previous exam questions related to Overfitting.",

    "Which questions were repeated in the previous exam papers?",

    "Give me the 5 most important Machine Learning questions based on previous papers.",

    "Explain Overfitting using the uploaded study notes."
]

In [ ]:
if not documents_uploaded:
    print("Please upload and process PDFs before running evaluation.")
else:
    faithfulness_scores = []
    answer_relevance_scores = []
    context_relevance_scores = []

    for query in evaluation_tasks:

        result = docu_chat(query)

        evaluation = llm_judge(
            query,
            result["context_used"],
            result["answer"]
        )

        faithfulness_scores.append(evaluation["faithfulness"])
        answer_relevance_scores.append(
            evaluation["answer_relevance"]
        )
        context_relevance_scores.append(
            evaluation["context_relevance"]
        )

    avg_faithfulness = sum(faithfulness_scores) / len(faithfulness_scores)
    avg_answer_relevance = sum(answer_relevance_scores) / len(answer_relevance_scores)
    avg_context_relevance = sum(context_relevance_scores) / len(context_relevance_scores)

    overall_score = (
        avg_faithfulness +
        avg_answer_relevance +
        avg_context_relevance
    ) / 3

    print("Evaluation Completed!\n")
    print("Average Faithfulness:", round(avg_faithfulness, 2), "/ 5")
    print("Average Answer Relevance:", round(avg_answer_relevance, 2), "/ 5")
    print("Average Context Relevance:", round(avg_context_relevance, 2), "/ 5")
    print("Overall RAG Score:", round(overall_score, 2), "/ 5")

Please upload and process PDFs before running evaluation.
